In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW expense_tracker_catalog.expense_tracker_schema.expense_gold AS
WITH user_ranking AS (
    SELECT 
        payment_method,
        user_id,
        SUM(expense_amount) AS total_amount,
        ROW_NUMBER() OVER(
            PARTITION BY payment_method
            ORDER BY SUM(expense_amount) DESC
        ) AS ranking
    FROM expense_tracker_catalog.expense_tracker_schema.expense_silver
    GROUP BY payment_method, user_id
)

SELECT 
    es.payment_method,
    COUNT(es.payment_method) AS total_transactions,
    SUM(es.expense_amount) AS total_amount,
    COUNT(DISTINCT es.user_id) AS total_users,
    MAX(es.expense_amount) AS max_transaction,
    ur.user_id AS top_user,
    ur.total_amount AS top_expense
FROM expense_tracker_catalog.expense_tracker_schema.expense_silver es
LEFT JOIN user_ranking ur ON ur.payment_method=es.payment_method WHERE ur.ranking=1 
GROUP BY 
    es.payment_method, ur.user_id, ur.total_amount;
